In [0]:
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.sql.functions import regexp_replace
from pyspark.sql.functions import trim
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.window import Window


In [0]:
tax_regime_path = 'tax_regime_silver.tax_regime_data'
tax_regime_df_schema = StructType([
  StructField('ano', StringType(), True),
  StructField('cnpj', StringType(), True),
  StructField('cnpj_da_scp', StringType(), True),
  StructField('forma_de_tributacao', StringType(), True),
  StructField('quantidade_de_escrituracoes', StringType(), True)
])



In [0]:
## -------Reading lucro_presumido data-------

tax_regime_lucro_presumido = (spark.read
.option("InferSchema", "True")
.option("header", "true")
.csv('/Volumes/workspace/cnpjs_schema/tax_regime_data/bronze/Lucro Presumido/*.csv')).select(
    (F.col("cnpj").alias("CNPJ")), 
    (F.col("forma_de_tributacao").alias("TaxRegime")),
    (F.col("ano"))
  )


## -------Reading all data-------


tax_regime_df = (spark.read
.option("InferSchema", "True")
.option("header", "true")
.csv('/Volumes/workspace/cnpjs_schema/tax_regime_data/bronze/*.csv')).select(
    (F.col("cnpj").alias("CNPJ")), 
    (F.col("forma_de_tributacao").alias("TaxRegime")),
    (F.col("ano"))
)
## -------Union of all tax_regime-------
final_tax_regime_df = tax_regime_df.unionByName(tax_regime_lucro_presumido)

## --------Cleaning data-------

final_tax_regime_df = final_tax_regime_df.withColumn('CNPJ',trim(regexp_replace(F.col('CNPJ'), r'[./-]', '')))

window = Window.partitionBy("CNPJ").orderBy(F.desc("ano"))
final_tax_regime_df = final_tax_regime_df.withColumn("rank", F.row_number().over(window))
final_tax_regime_df = final_tax_regime_df.filter(F.col("rank") == 1).drop("rank", "ano")
## --------Saving data in silver table-------

final_tax_regime_df.write \
  .format("delta") \
  .mode("overwrite") \
  .option ("overwriteSchema", "true") \
  .saveAsTable(f"workspace.{tax_regime_path}")



In [0]:
spark.read.table(f"workspace.{tax_regime_path}").display()